# 02 LLM Baseline: Llama 3.1 8B No-context

Run a practical local baseline with `llama3.1:8b` through Ollama.


## 1.Imports and Configuration


In [11]:
from __future__ import annotations

import gc
import json
import os
import random
import time
# import urllib.error
# import urllib.request
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from tqdm import tqdm

try:
    import evaluate
except ImportError as exc:
    raise ImportError("Install evaluation dependencies with: pip install evaluate bert-score") from exc


In [13]:
def load_env_file(path: Path) -> None:
    """Load simple KEY=VALUE entries from a local .env file if present."""
    if not path.exists():
        return

    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_env_file(PROJECT_ROOT / ".env")

SEED = 42
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434")
EVAL_LIMIT = 20

GENERATION_CONFIG = {
    "max_new_tokens": 128,
    "temperature": 0.2,
    "top_p": 0.9,
    "do_sample": False,
}

OLLAMA_OPTIONS = {
    "num_predict": GENERATION_CONFIG["max_new_tokens"],
    "temperature": GENERATION_CONFIG["temperature"],
    "top_p": GENERATION_CONFIG["top_p"],
    "seed": SEED,
}

RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PREDICTION_PATH = RESULTS_DIR / "llama_baseline_predictions.csv"

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

print(f"The generation parameters are fixed and logged.")


The generation parameters are fixed and logged.


## 2.Load Dataset


In [20]:
DATA_DIR = PROJECT_ROOT / "data" / "sentence_no_context" / "processed"
DATA_PATHS = {
    "train": DATA_DIR / "train.csv",
    "val": DATA_DIR / "val.csv",
    "test": DATA_DIR / "test.csv",
}

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    rename_map = {}
    if "complex" not in df.columns and "input_text" in df.columns:
        rename_map["input_text"] = "complex"
    if "simple" not in df.columns and "target_text" in df.columns:
        rename_map["target_text"] = "simple"

    df = df.rename(columns=rename_map).copy()
    required_columns = {"complex", "simple"}
    missing_columns = sorted(required_columns - set(df.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    df["complex"] = df["complex"].fillna("").astype(str).str.strip()
    df["simple"] = df["simple"].fillna("").astype(str).str.strip()
    return df[df["complex"].ne("") & df["simple"].ne("")].reset_index(drop=True)

def load_split(split_name: str) -> pd.DataFrame:
    path = DATA_PATHS[split_name]
    if not path.exists():
        raise FileNotFoundError(f"Missing {split_name} split: {path}")
    df = pd.read_csv(path)
    df = normalize_columns(df)
    print(f"Loaded {split_name}: {len(df):,} rows")
    return df

train_df = load_split("train")
val_df = load_split("val")
test_df = load_split("test")

EVAL_LIMIT = 892
test_eval_df = test_df.head(EVAL_LIMIT).reset_index(drop=True)

# Preview only the input column used by the LLM.
test_eval_df[["complex"]].head()


Loaded train: 6,742 rows
Loaded val: 984 rows
Loaded test: 892 rows


,complex
0,Computer reminders achieved a median improveme...
1,In the eight comparisons that reported dichoto...
2,A minority of interventions showed larger effe...
3,Further research must identify design features...
4,The oral anticoagulant was a vitamin K antagon...


## 3.Prompt Design



In [ ]:
PROMPT_TEMPLATE = """You are an expert in biomedical text simplification.

Rewrite the biomedical sentence for a general audience.

Rules:

* Preserve the original meaning.
* Use clear and simple language.
* Replace medical, scientific, or technical terms with simpler alternatives whenever possible.
* Remove unnecessary statistical details unless they are essential for understanding the main finding.
* Do not add information that is not present in the original sentence.
* Output exactly one simplified sentence.

Sentence:
{complex_sentence}

Simplified sentence:"""

def build_prompt(complex_sentence: str) -> str:
    return PROMPT_TEMPLATE.format(complex_sentence=complex_sentence.strip())

print("Built prompt for example sentence.")

Built prompt for example sentence.


## 4.Load Llama 3.1 8B via Ollama



In [23]:
def ollama_request(path: str, payload: dict[str, Any] | None = None, timeout: int = 120) -> dict[str, Any]:
    """Call the local Ollama HTTP API."""
    url = f"{OLLAMA_URL}{path}"
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    try:
        with urllib.request.urlopen(request, timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.URLError as exc:
        raise RuntimeError(
            "Could not reach Ollama. Make sure the Ollama app/server is running, "
            "then run `ollama list` in a terminal."
        ) from exc


def list_ollama_models() -> list[str]:
    response = ollama_request("/api/tags", timeout=15)
    return [model["name"] for model in response.get("models", [])]


def ensure_ollama_model(model_name: str) -> None:
    available_models = list_ollama_models()
    if model_name not in available_models:
        available = ", ".join(available_models) if available_models else "no local models"
        raise RuntimeError(
            f"Ollama model `{model_name}` is not installed. Available: {available}. "
            f"Install it with: ollama pull {model_name}"
        )
    print(f"Using Ollama model: {model_name}")


ensure_ollama_model(OLLAMA_MODEL)


Using Ollama model: llama3.1:8b


## 5.Inference Pipeline



In [24]:
def clean_prediction(text: str) -> str:
    text = text.strip()
    prefixes = ["Simplified sentence:", "Simplified:", "Answer:"]
    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()
    return " ".join(text.split())


def generate_prediction(complex_sentence: str) -> str:
    prompt = build_prompt(complex_sentence)
    response = ollama_request(
        "/api/generate",
        payload={
            "model": OLLAMA_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": OLLAMA_OPTIONS,
        },
        timeout=300,
    )
    return clean_prediction(response.get("response", ""))


def generate_predictions(input_df: pd.DataFrame) -> pd.DataFrame:
    """Generate predictions using only the complex sentence column."""
    if list(input_df.columns) != ["complex"]:
        raise ValueError("LLM input must contain only the complex sentence column.")

    results = input_df.copy()
    predictions = []
    start_time = time.time()

    for idx, sentence in tqdm(enumerate(results["complex"]), total=len(results), desc="Generating"):
        try:
            prediction = generate_prediction(sentence)
            if not prediction:
                prediction = "[EMPTY_OUTPUT]"
        except Exception as exc:
            prediction = "[GENERATION_FAILED]"
            print(f"Generation failed at row {idx}: {exc}")
        predictions.append(prediction)

    elapsed = time.time() - start_time
    print(f"Generated {len(results):,} examples in {elapsed:.1f}s")

    results["prediction"] = predictions
    return results


inference_inputs = test_eval_df[["complex"]].copy()
predictions_only_df = generate_predictions(inference_inputs)

prediction_df = test_eval_df.copy()
prediction_df["prediction"] = predictions_only_df["prediction"].values
prediction_df.to_csv(PREDICTION_PATH, index=False)
print(f"Saved predictions to: {PREDICTION_PATH.relative_to(PROJECT_ROOT)}")
prediction_df.head()


Generating:   0%|          | 3/892 [00:17<1:26:56,  5.87s/it]


KeyboardInterrupt: 

## 6.Evaluation

- **SARI** evaluates simplification edits by comparing the source, prediction, and reference.
- **BLEU** measures n-gram overlap between prediction and reference.
- **BERTScore** measures semantic similarity using contextual embeddings.

SARI is usually more informative for simplification than BLEU, while BERTScore gives a softer semantic match signal.

In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import compute_metrics

metrics_summary = compute_metrics(prediction_df)
display(metrics_summary)


## 7.Analysis



In [8]:
example_columns = ["complex", "simple", "prediction"]
prediction_df[example_columns].sample(n=min(10, len(prediction_df)), random_state=SEED)

,complex,simple,prediction
361,Newborn skin or cord cleansing with chlorhexid...,Chlorhexidine cord cleansing compared to dry c...,Using a special soap called chlorhexidine to c...
73,The question as to whether people with AMD sho...,The overall conclusion of this review is that ...,Researchers haven't found out if taking Ginkgo...
374,"In comparison, in a subgroup of 12 studies inc...","By contrast, in studies including participants...",Among people aged 16 and older who survived th...
155,Three of the four studies included neonates bo...,Three trials included neonates born at and bey...,Most of these studies looked at babies who wer...
104,This result should be interpreted with caution...,"However, there was significant variability in ...",The results may not be entirely reliable becau...
394,Only two trials were at low risk of bias.,The majority of the trials were at high risk o...,Two studies had a good chance of giving accura...
377,One study presented a prevalence for bone canc...,One study in bone cancer survivors reported no...,Out of a large group of people who survived bo...
124,Three participants in the steroid group of one...,Three participants in the steroid group of one...,"In one study, three people taking steroids exp..."
68,This review could not derive clear evidence fr...,Prevention of HBV recurrence following liver t...,This study found no strong proof that a certai...
450,Many trials were at high risk of bias reflecti...,Many trials used flawed methods that might hav...,The studies had a high chance of being unfair ...


In [27]:
# Optional cleanup after inference/evaluation.
gc.collect()


0